# Ch 3　開發環境、套件與版本策略

> 這份 notebook 幫你把「可重現的開發環境」一次設好，並跑通第一個 agent。

## 3.1–3.2　安裝與版本檢查

本書安裝一律用 uv；在 notebook 裡用 `!uv pip install` 即可（你照 Ch3 用 uv 專案啟動 Jupyter 的話，這會裝進專案環境）。重點是：**裝對版本**。

In [ ]:
# 三件套：langgraph（runtime）、langchain（framework）、deepagents（harness）+ Anthropic 整合
!uv pip install -q langchain langgraph deepagents langchain-google-genai

In [1]:
import sys
import langchain

print("Python :", sys.version.split()[0])    # 需要 3.10+
print("langchain:", langchain.__version__)     # 應該是 1.x.x —— 不是 1.x 就會踩到舊 API 的雷
assert sys.version_info >= (3, 10), "請升級到 Python 3.10 以上"

Python : 3.14.3
langchain: 1.3.1


## 3.3　設定 API key

別把金鑰寫死進程式碼。這裡用環境變數；正式專案請用 .env 或機密管理服務。

In [ ]:
import os, getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY：")

## 3.5　跑通你的第一個 agent

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """查詢指定城市的天氣。"""      # docstring = 給模型看的工具說明，別省略
    return f"{city} 一直都是晴天！"

agent = create_agent(
    model="google_genai:gemini-2.5-flash",        # 也可寫 "openai:gpt-5.4" 等，換 provider 只是換字串
    tools=[get_weather],
    system_prompt="你是個樂於助人的助理。",
)

result = agent.invoke({"messages": [{"role": "user", "content": "舊金山天氣如何？"}]})
print(result["messages"][-1].content_blocks)   # 看到舊金山天氣 = 環境通了，恭喜！

## 3.6　開啟 LangSmith tracing（強烈建議）

設三個環境變數，**完全不用改程式碼**，就能在 LangSmith 上「看見」agent 的每一步。學習階段這是建立直覺最快的方式。

In [ ]:
import os
# 想開 tracing 就把下面取消註解、填入你的 LangSmith key：
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "你的-langsmith-key"
# os.environ["LANGSMITH_PROJECT"] = "langgraph-deepagents-book"   # 選填，用來歸類專案

# 設好之後，重跑上一格的 agent.invoke(...)，到 smith.langchain.com 就能看到完整 trace。
print("tracing 設定示意，取消註解並填 key 後重跑上一格即可。")

## 3.7　版本鎖定的精神

在改版頻繁的 LangChain 生態，**把版本鎖死、把 lock 檔納入版控**，是讓程式碼三個月後還能跑的基本功。書裡每章開頭都標版本，正是這個原則。

In [ ]:
# 把你目前環境的關鍵版本印出來，貼進 requirements.txt / pyproject.toml 鎖起來。
import importlib.metadata as m
for pkg in ["langchain", "langgraph", "deepagents", "langchain-google-genai"]:
    try:
        print(f"{pkg}=={m.version(pkg)}")
    except Exception:
        print(f"{pkg}: (未安裝)")